In [28]:
import os
import random
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [29]:
# Path setup
BASE_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATASET_PATH = os.path.join(BASE_PATH, "data", "raw")

print(DATASET_PATH)

d:\VIT Personal\TrustFilterAI\ml\counterfeit_detection\data\raw


In [30]:
# Load dataset
classes = ["genuine", "fake"]
data = []

for brand in os.listdir(DATASET_PATH):
    brand_path = os.path.join(DATASET_PATH, brand)
    
    if not os.path.isdir(brand_path):
        continue
    
    for label in classes:
        label_path = os.path.join(brand_path, label)
        
        if not os.path.exists(label_path):
            continue
        
        for img_name in os.listdir(label_path):
            img_path = os.path.join(label_path, img_name)
            
            data.append({
                "image_path": img_path,
                "label": 0 if label == "genuine" else 1,
                "brand": brand
            })

df = pd.DataFrame(data)
df["brand_id"] = df["brand"].astype("category").cat.codes
print("Total images:", len(df))
df.head()

Total images: 4000


,image_path,label,brand,brand_id
0,d:\VIT Personal\TrustFilterAI\ml\counterfeit_d...,0,Audemars Piguet,0
1,d:\VIT Personal\TrustFilterAI\ml\counterfeit_d...,0,Audemars Piguet,0
2,d:\VIT Personal\TrustFilterAI\ml\counterfeit_d...,0,Audemars Piguet,0
3,d:\VIT Personal\TrustFilterAI\ml\counterfeit_d...,0,Audemars Piguet,0
4,d:\VIT Personal\TrustFilterAI\ml\counterfeit_d...,0,Audemars Piguet,0


In [31]:
# Define transformations
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [32]:
# Create dataset class
class WatchDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]["image_path"]
        label = self.df.iloc[idx]["label"]

        try:
            image = Image.open(img_path).convert("RGB")
        except:
            new_idx = random.randint(0, len(self.df)-1)
            return self.__getitem__(new_idx)

        if self.transform:
            image = self.transform(image)

        brand = self.df.iloc[idx]["brand_id"]
        return image, label, brand

In [33]:
# Split dataset
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [34]:
# Create dataset
train_dataset = WatchDataset(train_df, transform=train_transform)
val_dataset = WatchDataset(val_df, transform=val_transform)

In [35]:
# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [36]:
# Test dataloader
images, labels, brands = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Labels:", labels[:10])
print("Brands:", brands[:10])
print(df["label"].value_counts())

Image batch shape: torch.Size([32, 3, 224, 224])
Labels: tensor([1, 0, 1, 0, 0, 0, 1, 0, 0, 1])
Brands: tensor([0, 7, 9, 3, 2, 8, 4, 7, 1, 2], dtype=torch.int8)
label
0    2000
1    2000
Name: count, dtype: int64
